[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/skyexry/urban-mobility-forecast/blob/main/notebooks/05_train_eval.ipynb)

# 05 — Training & Evaluation

Train and compare four models on the 100-station CitiBike demand forecasting task:
- **Historical Average** — statistical baseline
- **LSTM** — pure time-series RNN baseline
- **TCN-only** — pure time-series deep baseline (no spatial)
- **STGNN** — full spatio-temporal model

Metrics: MAE, RMSE, MAPE on held-out test set (inverse transformed to original scale).

## Setup

In [1]:
!git clone https://github.com/skyexry/urban-mobility-forecast.git 2>/dev/null || git -C urban-mobility-forecast pull

In [2]:
import sys
from google.colab import drive
drive.mount('/content/drive')

!pip install torch-geometric -q
sys.path.append('/content/urban-mobility-forecast')

Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 43.4 MB/s eta 0:00:00


In [3]:
import importlib, numpy as np, pandas as pd
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import joblib, warnings
warnings.filterwarnings('ignore')

def load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

features_mod = load_module('features', '/content/urban-mobility-forecast/preprocessing/features.py')
from model.stconv import build_edge_index
from model.stgnn import STGNN
from model.tcn import TCNBlock

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

Device: cuda


## 1. Load Data

Load the filtered demand time series, station metadata, and pre-built graph from Drive.

In [4]:
df       = pd.read_parquet('/content/drive/MyDrive/citibike/hourly_demand_filtered.parquet')
stations = pd.read_parquet('/content/drive/MyDrive/citibike/stations_final.parquet')
df['hour'] = pd.to_datetime(df['hour'])

edge_index = torch.tensor(np.load('/content/drive/MyDrive/citibike/edge_index.npy'), dtype=torch.long).to(DEVICE)
edge_weight = torch.tensor(np.load('/content/drive/MyDrive/citibike/edge_weight.npy'), dtype=torch.float).to(DEVICE)

print(f'Stations   : {df["start_station_id"].nunique()}')
print(f'Date range : {df["hour"].min()} → {df["hour"].max()}')
print(f'edge_index : {edge_index.shape}')

Stations   : 100
Date range : 2024-01-01 00:00:00 → 2025-12-31 23:00:00
edge_index : torch.Size([2, 656])


## 2. Build Demand Matrix & Time Features

Pivot sparse data into dense `(T, N)` matrix and generate cyclic time encodings.

In [5]:
station_ids = stations['start_station_id'].tolist()
demand_matrix, hours = features_mod.build_demand_matrix(df, station_ids)
time_feats = features_mod.build_time_features(pd.Series(hours))
print(f'demand_matrix : {demand_matrix.shape}')
print(f'time_features : {time_feats.shape}')

demand_matrix : (16808, 100)
time_features : (16808, 6)


## 3. Train / Val / Test Split

Split chronologically (70/15/15) to avoid data leakage.
Scaler is fit **only on training data**, then applied to val and test.

In [6]:
T = demand_matrix.shape[0]
split1 = int(T * 0.70)
split2 = int(T * 0.85)

train_demand = demand_matrix[:split1]
val_demand   = demand_matrix[split1:split2]
test_demand  = demand_matrix[split2:]

train_time = time_feats[:split1]
val_time   = time_feats[split1:split2]
test_time  = time_feats[split2:]

# Fit scaler on train only (normalize_demand does log1p internally)
normalized_train, scaler = features_mod.normalize_demand(train_demand)

# Apply log1p first, then scaler.transform (mirrors normalize_demand internals)
normalized_val  = scaler.transform(np.log1p(val_demand).reshape(-1,1)).reshape(val_demand.shape)
normalized_test = scaler.transform(np.log1p(test_demand).reshape(-1,1)).reshape(test_demand.shape)

joblib.dump(scaler, '/content/drive/MyDrive/citibike/scaler.pkl')

print(f'Train: {train_demand.shape[0]} steps ({train_demand.shape[0]/24:.0f} days)')
print(f'Val  : {val_demand.shape[0]} steps ({val_demand.shape[0]/24:.0f} days)')
print(f'Test : {test_demand.shape[0]} steps ({test_demand.shape[0]/24:.0f} days)')

Train: 11765 steps (490 days)
Val  : 2521 steps (105 days)
Test : 2522 steps (105 days)


## 4. Sliding Windows & DataLoaders

Slide a 72-hour input / 72-hour output window over each split. Wrap in PyTorch Dataset and DataLoader.

In [7]:
INPUT_WINDOW  = 72
OUTPUT_WINDOW = 72
BATCH_SIZE    = 32

x_demand_train, x_time_train, y_train = features_mod.build_sliding_windows(normalized_train, train_time, INPUT_WINDOW, OUTPUT_WINDOW)
x_demand_val,   x_time_val,   y_val   = features_mod.build_sliding_windows(normalized_val,   val_time,   INPUT_WINDOW, OUTPUT_WINDOW)
x_demand_test,  x_time_test,  y_test  = features_mod.build_sliding_windows(normalized_test,  test_time,  INPUT_WINDOW, OUTPUT_WINDOW)

class CitiBikeDataset(Dataset):
    def __init__(self, x_demand, x_time, y):
        self.x_demand = torch.tensor(x_demand, dtype=torch.float32)
        self.x_time   = torch.tensor(x_time,   dtype=torch.float32)
        self.y        = torch.tensor(y,         dtype=torch.float32)
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.x_demand[i], self.x_time[i], self.y[i]

train_loader = DataLoader(CitiBikeDataset(x_demand_train, x_time_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(CitiBikeDataset(x_demand_val,   x_time_val,   y_val),   batch_size=BATCH_SIZE)
test_loader  = DataLoader(CitiBikeDataset(x_demand_test,  x_time_test,  y_test),  batch_size=BATCH_SIZE)
print(f'Train batches: {len(train_loader)}, Val: {len(val_loader)}, Test: {len(test_loader)}')

Samples  : 11622
x_demand : (11622, 100, 72, 1)
x_time   : (11622, 72, 6)
y        : (11622, 100, 72, 1)
Samples  : 2378
x_demand : (2378, 100, 72, 1)
x_time   : (2378, 72, 6)
y        : (2378, 100, 72, 1)
Samples  : 2379
x_demand : (2379, 100, 72, 1)
x_time   : (2379, 72, 6)
y        : (2379, 100, 72, 1)
Train batches: 364, Val: 75, Test: 75


## 5. Training Utilities

Shared training loop, early stopping, and evaluation functions used by all models.

In [8]:
from tqdm.notebook import tqdm

def train_epoch(model, loader, optimizer, loss_fn, forward_fn):
    model.train()
    total = 0
    for x_d, x_t, y in loader:
        x_d, x_t, y = x_d.to(DEVICE), x_t.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        pred = forward_fn(model, x_d, x_t)
        loss = loss_fn(pred, y.squeeze(-1))
        loss.backward()
        optimizer.step()
        total += loss.item()
    return total / len(loader)

def eval_epoch(model, loader, loss_fn, forward_fn):
    model.eval()
    total = 0
    with torch.no_grad():
        for x_d, x_t, y in loader:
            x_d, x_t, y = x_d.to(DEVICE), x_t.to(DEVICE), y.to(DEVICE)
            pred = forward_fn(model, x_d, x_t)
            total += loss_fn(pred, y.squeeze(-1)).item()
    return total / len(loader)

def run_training(model, train_loader, val_loader, forward_fn,
                 lr=1e-3, epochs=100, patience=10, save_path=None):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn   = nn.L1Loss()
    best_val, wait, history = float('inf'), 0, []

    pbar = tqdm(range(1, epochs + 1), desc='Training', unit='epoch')
    for epoch in pbar:
        tr = train_epoch(model, train_loader, optimizer, loss_fn, forward_fn)
        vl = eval_epoch(model, val_loader, loss_fn, forward_fn)
        history.append((tr, vl))
        pbar.set_postfix({'train': f'{tr:.4f}', 'val': f'{vl:.4f}',
                          'best': f'{best_val:.4f}', 'patience': f'{wait}/{patience}'})
        if vl < best_val:
            best_val, wait = vl, 0
            if save_path: torch.save(model.state_dict(), save_path)
        else:
            wait += 1
            if wait >= patience:
                pbar.set_description(f'Early stop @ epoch {epoch}')
                break
    return history

def compute_metrics(y_true, y_pred, scaler):
    y_true = features_mod.inverse_transform_demand(y_true, scaler)
    y_pred = features_mod.inverse_transform_demand(y_pred, scaler)
    mae  = np.mean(np.abs(y_true - y_pred))
    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
    mask = y_true > 0
    mape = np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100
    return dict(MAE=mae, RMSE=rmse, MAPE=mape)

def get_predictions(model, loader, forward_fn):
    model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for x_d, x_t, y in tqdm(loader, desc='Predicting', leave=False):
            x_d, x_t = x_d.to(DEVICE), x_t.to(DEVICE)
            preds.append(forward_fn(model, x_d, x_t).cpu().numpy())
            trues.append(y.squeeze(-1).numpy())
    return np.concatenate(preds), np.concatenate(trues)

## 6. Baseline — Historical Average

Predicts the mean demand for each station and hour-of-day computed from the training set.
No learning required — serves as the lower bound.

In [9]:
# For each hour-of-day (0–23), compute mean demand per station over training set
N = normalized_train.shape[1]
T_train = normalized_train.shape[0]
hod_train = np.array([h.hour for h in hours[:T_train]])

ha_pred = np.zeros((24, N))
for h in range(24):
    mask = hod_train == h
    ha_pred[h] = normalized_train[mask].mean(axis=0)

# Evaluate on test set
hod_test = np.array([h.hour for h in hours[split2:]])
y_true_ha, y_pred_ha = [], []
for i in range(INPUT_WINDOW, len(normalized_test) - OUTPUT_WINDOW + 1):
    y_true_ha.append(normalized_test[i:i+OUTPUT_WINDOW])
    pred_hours = hod_test[i:i+OUTPUT_WINDOW]
    y_pred_ha.append(ha_pred[pred_hours])

y_true_ha = np.stack(y_true_ha)  # (samples, OUTPUT_WINDOW, N)
y_pred_ha = np.stack(y_pred_ha)

metrics_ha = compute_metrics(y_true_ha, y_pred_ha, scaler)
print('Historical Average:', {k: f"{v:.4f}" for k, v in metrics_ha.items()})

Historical Average: {'MAE': '13.5284', 'RMSE': '21.7451', 'MAPE': '68.0367'}


## 7. Baseline — LSTM

Per-node LSTM trained independently on each station's demand sequence.
Captures temporal patterns but has no spatial awareness.

In [10]:
class LSTMModel(nn.Module):
    def __init__(self, num_nodes, input_window, output_window, hidden=64, num_layers=2):
        super().__init__()
        self.num_nodes = num_nodes
        self.output_window = output_window
        # Shared LSTM across all nodes
        self.lstm = nn.LSTM(1, hidden, num_layers, batch_first=True, dropout=0.2)
        self.fc   = nn.Linear(hidden, output_window)

    def forward(self, x_demand, x_time=None):
        # x_demand: (batch, N, T, 1)
        b, N, T, _ = x_demand.shape
        x = x_demand.reshape(b * N, T, 1)
        out, _ = self.lstm(x)          # (b*N, T, hidden)
        out = self.fc(out[:, -1, :])   # (b*N, output_window)
        return out.reshape(b, N, self.output_window)

lstm_model = LSTMModel(num_nodes=100, input_window=INPUT_WINDOW, output_window=OUTPUT_WINDOW).to(DEVICE)
lstm_forward = lambda model, x_d, x_t: model(x_d, x_t)

print(f'LSTM parameters: {sum(p.numel() for p in lstm_model.parameters()):,}')
history_lstm = run_training(
    lstm_model, train_loader, val_loader, lstm_forward,
    save_path='/content/drive/MyDrive/citibike/lstm_best.pth'
)
lstm_model.load_state_dict(torch.load('/content/drive/MyDrive/citibike/lstm_best.pth'))
preds_lstm, trues_lstm = get_predictions(lstm_model, test_loader, lstm_forward)
metrics_lstm = compute_metrics(trues_lstm, preds_lstm, scaler)
print('LSTM:', {k: f"{v:.4f}" for k, v in metrics_lstm.items()})

LSTM parameters: 55,112


Training:   0%|          | 0/100 [00:00<?, ?epoch/s]

Predicting:   0%|          | 0/75 [00:00<?, ?it/s]

LSTM: {'MAE': '8.4688', 'RMSE': '13.7326', 'MAPE': '68.0431'}


## 8. Baseline — TCN-only

Same architecture as the temporal branch in STGNN, but without graph convolution.
Direct ablation: difference vs STGNN shows the value of spatial information.

In [ ]:
class TCNOnlyModel(nn.Module):
    def __init__(self, num_nodes, input_window, output_window,
                 tcn_channels=32, time_channels=6, num_layers=2):
        super().__init__()
        self.num_nodes = num_nodes
        self.output_window = output_window
        self.demand_tcn = TCNBlock(in_channels=1,            out_channels=tcn_channels, num_layers=num_layers)
        self.time_tcn   = TCNBlock(in_channels=time_channels, out_channels=tcn_channels, num_layers=num_layers)
        self.fc = nn.Linear(tcn_channels * 2, output_window)

    def forward(self, x_demand, x_time):
        b, N, T, _ = x_demand.shape
        x_d = x_demand.reshape(b * N, T, 1).permute(0, 2, 1)  # (b*N, 1, T)
        d_out = self.demand_tcn(x_d)[:, :, -1]                # (b*N, tcn_ch)

        x_t = x_time.permute(0, 2, 1)                         # (batch, 6, T)
        t_out = self.time_tcn(x_t)[:, :, -1]                  # (batch, tcn_ch)
        t_out = t_out.unsqueeze(1).expand(-1, N, -1).reshape(b * N, -1)

        fused = torch.cat([d_out, t_out], dim=-1)
        out = self.fc(fused).reshape(b, N, self.output_window)
        return out

tcn_model = TCNOnlyModel(num_nodes=100, input_window=INPUT_WINDOW, output_window=OUTPUT_WINDOW).to(DEVICE)
tcn_forward = lambda model, x_d, x_t: model(x_d, x_t)

print(f'TCN-only parameters: {sum(p.numel() for p in tcn_model.parameters()):,}')
history_tcn = run_training(
    tcn_model, train_loader, val_loader, tcn_forward,
    save_path='/content/drive/MyDrive/citibike/tcn_best.pth'
)
tcn_model.load_state_dict(torch.load('/content/drive/MyDrive/citibike/tcn_best.pth'))
preds_tcn, trues_tcn = get_predictions(tcn_model, test_loader, tcn_forward)
metrics_tcn = compute_metrics(trues_tcn, preds_tcn, scaler)
print('TCN-only:', {k: f"{v:.4f}" for k, v in metrics_tcn.items()})

## 9. STGNN — Full Spatio-Temporal Model

Full model with ChebConv graph encoding + TCN temporal encoding + Self-Attention decoder.
Incorporates both spatial dependencies between stations and temporal patterns.

In [ ]:
stgnn_model = STGNN(num_nodes=100, input_window=INPUT_WINDOW, output_window=OUTPUT_WINDOW).to(DEVICE)
stgnn_forward = lambda model, x_d, x_t: model(x_d, x_t, edge_index, edge_weight)

print(f'STGNN parameters: {sum(p.numel() for p in stgnn_model.parameters()):,}')
history_stgnn = run_training(
    stgnn_model, train_loader, val_loader, stgnn_forward,
    save_path='/content/drive/MyDrive/citibike/stgnn_best.pth'
)
stgnn_model.load_state_dict(torch.load('/content/drive/MyDrive/citibike/stgnn_best.pth'))
preds_stgnn, trues_stgnn = get_predictions(stgnn_model, test_loader, stgnn_forward)
metrics_stgnn = compute_metrics(trues_stgnn, preds_stgnn, scaler)
print('STGNN:', {k: f"{v:.4f}" for k, v in metrics_stgnn.items()})

## 10. Loss Curves

Visualize training and validation loss for each learned model.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, history, name in zip(axes,
    [history_lstm, history_tcn, history_stgnn],
    ['LSTM', 'TCN-only', 'STGNN']):
    tr = [h[0] for h in history]
    vl = [h[1] for h in history]
    ax.plot(tr, label='Train')
    ax.plot(vl, label='Val')
    ax.set_title(name)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('MAE Loss')
    ax.legend()
plt.tight_layout()
plt.show()

## 11. Results Comparison

Summary table comparing all four models on the test set (original demand scale).

In [ ]:
results = pd.DataFrame([
    {'Model': 'Historical Average', **metrics_ha},
    {'Model': 'LSTM',              **metrics_lstm},
    {'Model': 'TCN-only',          **metrics_tcn},
    {'Model': 'STGNN',             **metrics_stgnn},
]).set_index('Model').round(4)

print(results.to_string())
results